# 01 — Generate NIL Sponsorship Data

Generates synthetic data for a college athlete NIL (Name, Image, Likeness)
sponsorship platform using Faker with a fixed seed for reproducibility.

**Entities:**
- **Athletes** — 200 college athletes across 10 schools, 5 conferences, 6 sports
- **Sponsors** — 50 brands across 5 industries
- **Deals** — 300 NIL contracts connecting athletes to sponsors
- **Campaigns** — 800 marketing campaign performance records

Files land as newline-delimited JSON in a UC Volume at:
`/Volumes/{catalog}/{schema}_bronze/raw_data/{entity}/*.json`

In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from dotenv import load_dotenv
import os, json, random, uuid
from datetime import datetime, timedelta
from faker import Faker

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()
w = WorkspaceClient()

fake = Faker()
Faker.seed(42)
random.seed(42)

In [2]:
UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "nil_sponsorships")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"

spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")
print(f"Volume ready: {VOLUME_PATH}")

Volume ready: /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data


In [3]:
# Reference data
SCHOOLS = {
    "SEC":     ["University of Alabama", "University of Georgia"],
    "Big Ten": ["Ohio State University", "University of Michigan"],
    "Big 12":  ["University of Texas", "University of Oklahoma"],
    "ACC":     ["Clemson University", "Duke University"],
    "Pac-12":  ["University of Oregon", "USC"],
}

SPORTS_POSITIONS = {
    "Football":           ["QB", "RB", "WR", "TE", "OL", "DL", "LB", "CB", "S", "K"],
    "Men's Basketball":   ["PG", "SG", "SF", "PF", "C"],
    "Women's Basketball": ["PG", "SG", "SF", "PF", "C"],
    "Baseball":           ["P", "C", "1B", "2B", "SS", "3B", "OF"],
    "Softball":           ["P", "C", "1B", "2B", "SS", "3B", "OF"],
    "Soccer":             ["GK", "DEF", "MID", "FWD"],
}

YEARS = ["Freshman", "Sophomore", "Junior", "Senior"]

INDUSTRIES = ["Apparel", "Food & Beverage", "Technology", "Automotive", "Finance"]

BUDGET_TIERS = ["Starter", "Growth", "Enterprise"]

REGIONS = ["National", "Southeast", "Midwest", "West Coast", "Northeast", "Southwest"]

DEAL_TYPES = [
    "Social Media Post", "Autograph Signing", "Brand Ambassador",
    "Merchandise Licensing", "Event Appearance", "Content Creation",
]

PLATFORMS = ["Instagram", "TikTok", "Twitter", "YouTube", "In-Person"]

print(f"Schools: {sum(len(v) for v in SCHOOLS.values())}")
print(f"Sports: {len(SPORTS_POSITIONS)}")
print(f"Industries: {len(INDUSTRIES)}")

Schools: 10
Sports: 6
Industries: 5


## Generate Athletes

In [4]:
athletes = []
for i in range(200):
    conference = random.choice(list(SCHOOLS.keys()))
    school = random.choice(SCHOOLS[conference])
    sport = random.choice(list(SPORTS_POSITIONS.keys()))
    position = random.choice(SPORTS_POSITIONS[sport])

    # Social following correlates loosely with sport popularity
    base_followers = {"Football": 50000, "Men's Basketball": 40000,
                      "Women's Basketball": 30000, "Baseball": 15000,
                      "Softball": 12000, "Soccer": 10000}
    base = base_followers.get(sport, 10000)

    athletes.append({
        "athlete_id": str(uuid.uuid4()),
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.email(),
        "phone": fake.phone_number(),
        "school": school,
        "conference": conference,
        "sport": sport,
        "position": position,
        "year": random.choice(YEARS),
        "instagram_followers": int(random.gauss(base, base * 0.6)),
        "tiktok_followers": int(random.gauss(base * 0.8, base * 0.5)),
        "twitter_followers": int(random.gauss(base * 0.3, base * 0.2)),
    })

# Ensure no negative follower counts
for a in athletes:
    for col in ["instagram_followers", "tiktok_followers", "twitter_followers"]:
        a[col] = max(a[col], 100)

print(f"Generated {len(athletes)} athletes")
print(f"Sample: {athletes[0]['first_name']} {athletes[0]['last_name']} — {athletes[0]['sport']} at {athletes[0]['school']}")

Generated 200 athletes
Sample: Danielle Johnson — Soccer at University of Alabama


## Generate Sponsors

In [5]:
BRAND_PREFIXES = {
    "Apparel": ["Peak", "Summit", "Stride", "Flex", "Iron", "Apex", "Trail", "Bolt", "Ridge", "Core"],
    "Food & Beverage": ["Fresh", "Harvest", "Pulse", "Fuel", "Bloom", "Crisp", "Zest", "Grain", "Vital", "Maple"],
    "Technology": ["Nexus", "Pixel", "Cipher", "Quantum", "Arc", "Flux", "Logic", "Nova", "Prism", "Helix"],
    "Automotive": ["Drive", "Torque", "Velocity", "Cruiser", "Axle", "Piston", "Drift", "Turbo", "Ignite", "Ranger"],
    "Finance": ["Vault", "Ledger", "Mint", "Pinnacle", "Trust", "Summit", "Sterling", "Capital", "Merit", "Anchor"],
}
BRAND_SUFFIXES = ["Athletics", "Co", "Labs", "Group", "Brand", "Sports", "Collective", "Ventures", "HQ", "Nation"]

sponsors = []
used_names = set()
for i in range(50):
    industry = INDUSTRIES[i % len(INDUSTRIES)]
    while True:
        name = f"{random.choice(BRAND_PREFIXES[industry])} {random.choice(BRAND_SUFFIXES)}"
        if name not in used_names:
            used_names.add(name)
            break

    sponsors.append({
        "sponsor_id": str(uuid.uuid4()),
        "brand_name": name,
        "industry": industry,
        "budget_tier": random.choice(BUDGET_TIERS),
        "region": random.choice(REGIONS),
        "contact_name": fake.name(),
        "contact_email": fake.company_email(),
    })

print(f"Generated {len(sponsors)} sponsors")
print(f"Sample: {sponsors[0]['brand_name']} ({sponsors[0]['industry']}, {sponsors[0]['budget_tier']})")

Generated 50 sponsors
Sample: Trail HQ (Apparel, Growth)


## Generate Deals

In [6]:
DEAL_AMOUNT_RANGES = {
    "Social Media Post":     (500, 15000),
    "Autograph Signing":     (1000, 10000),
    "Brand Ambassador":      (5000, 75000),
    "Merchandise Licensing": (2000, 50000),
    "Event Appearance":      (2000, 25000),
    "Content Creation":      (1000, 20000),
}

STATUSES = ["Pending", "Active", "Completed", "Cancelled"]
STATUS_WEIGHTS = [0.15, 0.40, 0.35, 0.10]

deals = []
for i in range(300):
    athlete = random.choice(athletes)
    sponsor = random.choice(sponsors)
    deal_type = random.choice(DEAL_TYPES)
    lo, hi = DEAL_AMOUNT_RANGES[deal_type]

    # Higher-profile athletes command higher deals
    total_followers = (athlete["instagram_followers"] +
                       athlete["tiktok_followers"] +
                       athlete["twitter_followers"])
    follower_multiplier = min(total_followers / 50000, 3.0)
    amount = round(random.uniform(lo, hi) * max(follower_multiplier, 0.5), 2)

    start_date = fake.date_between(start_date="-6m", end_date="+3m")
    duration_days = random.randint(30, 365)
    end_date = start_date + timedelta(days=duration_days)

    deals.append({
        "deal_id": str(uuid.uuid4()),
        "athlete_id": athlete["athlete_id"],
        "sponsor_id": sponsor["sponsor_id"],
        "deal_type": deal_type,
        "deal_amount": amount,
        "status": random.choices(STATUSES, weights=STATUS_WEIGHTS, k=1)[0],
        "start_date": start_date.isoformat(),
        "end_date": end_date.isoformat(),
        "created_at": fake.date_time_between(start_date="-6m", end_date="now").isoformat(),
    })

print(f"Generated {len(deals)} deals")
total_value = sum(d["deal_amount"] for d in deals)
print(f"Total deal value: ${total_value:,.2f}")
print(f"Avg deal: ${total_value / len(deals):,.2f}")

Generated 300 deals
Total deal value: $4,901,686.77
Avg deal: $16,338.96


## Generate Campaigns

In [7]:
from datetime import date as dt_date

campaigns = []
today = datetime.now().date()
active_deals = [d for d in deals if d["status"] in ("Active", "Completed")]

for i in range(800):
    deal = random.choice(active_deals)
    platform = random.choice(PLATFORMS)

    # Impressions scale with platform type
    platform_base = {"TikTok": 50000, "Instagram": 30000, "YouTube": 20000,
                     "Twitter": 15000, "In-Person": 5000}
    base_impressions = platform_base.get(platform, 10000)
    impressions = max(int(random.gauss(base_impressions, base_impressions * 0.4)), 500)

    # Engagement rates vary by platform
    engagement_rate = {"TikTok": 0.06, "Instagram": 0.04, "YouTube": 0.03,
                       "Twitter": 0.02, "In-Person": 0.15}.get(platform, 0.03)
    engagements = int(impressions * random.uniform(engagement_rate * 0.5, engagement_rate * 1.5))
    clicks = int(engagements * random.uniform(0.1, 0.4))
    conversions = int(clicks * random.uniform(0.01, 0.08))

    deal_start = dt_date.fromisoformat(deal["start_date"])
    deal_end = dt_date.fromisoformat(deal["end_date"])
    campaign_end = min(deal_end, today)
    # Ensure start <= end
    if deal_start > campaign_end:
        campaign_end = deal_start + timedelta(days=30)

    campaign_date = fake.date_between(start_date=deal_start, end_date=campaign_end)

    campaigns.append({
        "campaign_id": str(uuid.uuid4()),
        "deal_id": deal["deal_id"],
        "platform": platform,
        "campaign_date": campaign_date.isoformat(),
        "impressions": impressions,
        "engagements": engagements,
        "clicks": clicks,
        "conversions": conversions,
        "spend_amount": round(random.uniform(100, 5000), 2),
    })

print(f"Generated {len(campaigns)} campaign records")
total_impressions = sum(c["impressions"] for c in campaigns)
print(f"Total impressions: {total_impressions:,}")

Generated 800 campaign records
Total impressions: 19,129,139


## Write to UC Volume

In [8]:
def write_ndjson(records, entity):
    """Write records as newline-delimited JSON to a UC Volume."""
    file_path = f"{VOLUME_PATH}/{entity}/{entity}.json"
    payload = "\n".join(json.dumps(r) for r in records).encode("utf-8")
    w.files.upload(file_path, payload, overwrite=True)
    size_mb = len(payload) / (1024 * 1024)
    print(f"  {entity}: {len(records):,} records ({size_mb:.2f} MB) -> {file_path}")


print("Writing to UC Volume...")
write_ndjson(athletes, "athletes")
write_ndjson(sponsors, "sponsors")
write_ndjson(deals, "deals")
write_ndjson(campaigns, "campaigns")
print("\nAll data written to Volume.")

Writing to UC Volume...
  athletes: 200 records (0.07 MB) -> /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/athletes/athletes.json
  sponsors: 50 records (0.01 MB) -> /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/sponsors/sponsors.json
  deals: 300 records (0.10 MB) -> /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/deals/deals.json
  campaigns: 800 records (0.20 MB) -> /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data/campaigns/campaigns.json

All data written to Volume.


In [9]:
print("=" * 60)
print("DATA GENERATION SUMMARY")
print("=" * 60)
print(f"  Athletes:   {len(athletes):,}")
print(f"  Sponsors:   {len(sponsors):,}")
print(f"  Deals:      {len(deals):,} (${sum(d['deal_amount'] for d in deals):,.2f} total)")
print(f"  Campaigns:  {len(campaigns):,} ({sum(c['impressions'] for c in campaigns):,} impressions)")
print(f"\nVolume path: {VOLUME_PATH}")
print("Ready for Bronze layer ingestion.")

DATA GENERATION SUMMARY
  Athletes:   200
  Sponsors:   50
  Deals:      300 ($4,901,686.77 total)
  Campaigns:  800 (19,129,139 impressions)

Volume path: /Volumes/alexander_booth/nil_sponsorships_bronze/raw_data
Ready for Bronze layer ingestion.
